In [1]:
import pandas as pd

In [40]:
canola_networks = []
canola_files = ['./data/canola_2023.csv', './data/canola_2024.csv', './data/canola_2025.csv']

soybean_networks = []
soybean_files = ['./data/soybean_2023.csv', './data/soybean_2024.csv', './data/soybean_2025.csv']

steel_networks = []
steel_files = ['./data/steel_2023.csv', './data/steel_2024.csv', './data/steel_2025.csv']

aluminum_networks = []
aluminum_files = ['./data/aluminum_2023.csv', './data/aluminum_2024.csv', './data/aluminum_2025.csv']

oil = []
oil_files = ['./data/oil_2023.csv', './data/oil_2024.csv', './data/oil_2025.csv']

In [41]:
tariff_files = canola_files + soybean_files + steel_files + aluminum_files + oil_files
dfs_normalized = []
for f in tariff_files:
    df = pd.read_csv(f)
    df = df.set_index('Economy/Tariffed Country')
    max_value = df.max(axis=None)
    df = df.map(lambda x: x/max_value)
    dfs_normalized.append(df)


In [42]:
df_normalized = dfs_normalized[0]
for i in range(1,len(dfs_normalized)):
    df_normalized = df_normalized.add(dfs_normalized[i])

In [43]:
df_normalized

,USA,China,Germany,Japan,India (2024),United Kingdom,France,Italy,Canada,Brazil,Spain,Mexico,South Korea,Australia,Turkey (2024),Indonesia (2024),Netherlands,Saudi Arabia,Poland
Economy/Tariffed Country,,,,,,,,,,,,,,,,,,,
USA,0.000000,1.317801,1.317801,1.317801,1.317801,1.317801,1.317801,1.317801,0.000000,0.717801,1.317801,0.000000,0.000000,0.000000,1.317801,0.717801,0.900000,1.317801,1.317801
China,5.355411,0.000000,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411,5.355411
Germany,4.267801,4.267801,0.000000,0.000000,1.500000,0.000000,0.000000,0.000000,0.000000,4.267801,0.000000,0.000000,0.000000,4.267801,0.000000,3.150000,0.000000,4.267801,0.000000
Japan,1.348216,1.208216,0.000000,0.000000,0.398216,0.000000,0.000000,0.000000,0.550000,1.348216,0.000000,0.000000,1.208216,0.550000,1.268216,0.398216,0.000000,1.348216,0.000000
India (2024),11.390667,11.390667,11.390667,11.390667,0.000000,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667,11.390667
United Kingdom,3.591688,3.591688,3.591688,3.591688,1.691688,2.000000,3.591688,3.591688,3.591688,3.591688,3.591688,3.591688,3.591688,3.591688,3.591688,1.691688,3.591688,3.591688,3.591688
France,4.267801,4.267801,0.000000,0.000000,1.500000,0.000000,0.000000,0.000000,0.000000,4.267801,0.000000,0.000000,0.000000,4.267801,0.000000,3.150000,0.000000,4.267801,0.000000
Italy,4.267801,4.267801,0.000000,0.000000,1.500000,0.000000,0.000000,0.000000,0.000000,4.267801,0.000000,0.000000,0.000000,4.267801,0.000000,3.150000,0.000000,4.267801,0.000000
Canada,0.000000,0.391688,0.000000,0.000000,0.391688,0.080000,0.000000,0.000000,0.000000,0.391688,0.000000,0.000000,0.235844,0.000000,0.391688,0.391688,0.000000,0.391688,0.000000


In [44]:
import networkx as nx
g = nx.DiGraph()
g_rev = nx.DiGraph()
# g_weight_filtered = nx.DiGraph()

for index, row in df_normalized.iterrows():
    for col in row.index:
        if row[col] != 0:
            g.add_edge(row.name, col, weight=row[col])
            g_rev.add_edge(col, row.name, weight=row[col])
            # if row[col] >= 1:
            #     g.add_edge(row.name, col, weight=row[col])

## PageRank (on reverse network)
High outdegree in original network correlates to high pagerank, means the country has high tariffs

In [47]:
pr_rev = nx.pagerank(g_rev)
pr_rev_sorted = {k: v for k, v in sorted(pr_rev.items(), key=lambda item: item[1], reverse=True)} # for sorting
pr_rev_sorted

{'India (2024)': 0.14755394453011272,
 'Saudi Arabia': 0.11297512391333084,
 'Turkey (2024)': 0.09372962568428853,
 'China': 0.08139083438432727,
 'Mexico': 0.07390823901752368,
 'Brazil': 0.07145575187592695,
 'Indonesia (2024)': 0.06991484649048321,
 'South Korea': 0.058443755729221725,
 'United Kingdom': 0.052716751188116265,
 'Germany': 0.028335993513102814,
 'France': 0.028335993513102814,
 'Italy': 0.028335993513102814,
 'Spain': 0.028335993513102814,
 'Netherlands': 0.028335993513102814,
 'Poland': 0.028335993513102814,
 'USA': 0.02212772538411629,
 'Australia': 0.017738562710917418,
 'Japan': 0.01678237514978363,
 'Canada': 0.011246502863234416}

## PageRank (on reverse network) - Normalized

In [46]:
pr_rev_normalized = {}
pr_max = max(pr_rev.values())
pr_min = min(pr_rev.values())
for u in pr_rev:
    pr_rev_normalized[u] = (pr_rev[u] - pr_min) / (pr_max - pr_min)

pr_rev_normalized

{'India (2024)': 1.0,
 'Saudi Arabia': 0.7463174409707648,
 'Turkey (2024)': 0.6051256029192784,
 'China': 0.5146038298665935,
 'Mexico': 0.4597088419238932,
 'Brazil': 0.4417165216836649,
 'Indonesia (2024)': 0.43041189028129756,
 'South Korea': 0.34625587780697,
 'United Kingdom': 0.30424053021427083,
 'Germany': 0.1253745976073221,
 'France': 0.1253745976073221,
 'Italy': 0.1253745976073221,
 'Spain': 0.1253745976073221,
 'Netherlands': 0.1253745976073221,
 'Poland': 0.1253745976073221,
 'USA': 0.07982852871286727,
 'Australia': 0.047628066144392495,
 'Japan': 0.04061313321453371,
 'Canada': 0.0}

In [17]:
def con_score(g):
    con_scores = {} # dictionary of con scores, indices are node names, values are con scores
    for u in g:
        score = 0
        for v in g:
            if u != v:
                common_out_neighbors = set(g.predecessors(u)).intersection(set(g.predecessors(u)))
                score = score + len(common_out_neighbors)
        con_scores[u] = score

    return con_scores

def normalized_con_score(g):
    con_scores = {} # dictionary of con scores, indices are node names, values are con scores
    for u in g:
        score = 0
        for v in g:
            if u != v:
                common_out_neighbors = set(g.predecessors(u)).intersection(set(g.predecessors(u)))
                score = score + len(common_out_neighbors)
        con_scores[u] = score

    max_con = max(con_scores.values())
    min_con = min(con_scores.values())

    normalized_con_scores = {} # dictionary of con scores, indices are node names, values are normalized con scores
    for u in con_scores:
        normalized_con_scores[u] = (con_scores[u] - min_con) / (max_con - min_con)
    
    return normalized_con_scores

## CON Scores

In [49]:
con_scores = con_score(g)
con_scores

{'USA': 288,
 'China': 306,
 'Germany': 198,
 'Japan': 180,
 'India (2024)': 306,
 'United Kingdom': 216,
 'France': 198,
 'Italy': 198,
 'Brazil': 324,
 'Spain': 198,
 'Turkey (2024)': 216,
 'Indonesia (2024)': 306,
 'Netherlands': 198,
 'Saudi Arabia': 324,
 'Poland': 198,
 'Canada': 198,
 'Mexico': 144,
 'South Korea': 180,
 'Australia': 288}

## CON Scores - Normalized

In [50]:
normalized_con_scores = normalized_con_score(g)
normalized_con_scores_sorted = {k: v for k, v in sorted(normalized_con_scores.items(), key=lambda item: item[1], reverse=True)} # for sorting
normalized_con_scores_sorted

{'Brazil': 1.0,
 'Saudi Arabia': 1.0,
 'China': 0.9,
 'India (2024)': 0.9,
 'Indonesia (2024)': 0.9,
 'USA': 0.8,
 'Australia': 0.8,
 'United Kingdom': 0.4,
 'Turkey (2024)': 0.4,
 'Germany': 0.3,
 'France': 0.3,
 'Italy': 0.3,
 'Spain': 0.3,
 'Netherlands': 0.3,
 'Poland': 0.3,
 'Canada': 0.3,
 'Japan': 0.2,
 'South Korea': 0.2,
 'Mexico': 0.0}